# Agon: Eos on a Colab A100: validation and core experiment

Runs the thesis experiments with exactly the same commands as the HPRC jobs (`scripts/experiment.py`), so runs from either place can be compared.

The setup (`PRESET`, set in step 2; `society30` by default): 64 slots in training (continuous worlds grow without limit) and 8 founders who reproduce on a 32 x 32 map at half-hour ticks (a cell is 2 km; `society` is the 250 x 250, 1-minute version, which the viability check showed a band cannot live in, and which costs 30x the compute per game day); survival is the only reward; given survival skills (the "caveman" slate); sleep, a cognitive map and felt needs; attachment (others' survival felt as one's own, grown from being kept alive by them, faster toward a like-tempered, fit partner, and through danger faced and hunts made together); 17 conditions, good and bad, inborn (heritable, with rarities), coming with age, triggered by what happens, or earned by use; and the **living world**: grass, forest, rock and water plus desert, tundra, marsh and caves, moving clouds with local rain and snow, shifting wind, and a 3-year climate cycle.

**Before you start:** Runtime → Change runtime type → **A100 GPU** (a T4 also works, just slower). Run the cells in order.

| Step | What it does | Time on an A100 (estimate) |
| --- | --- | --- |
| 1–3 | GPU check, load code, connect Drive | a few minutes |
| 4 | Benchmark: finds how many parallel worlds fit and how fast | about 10 minutes |
| 4b | Viability check: can a competent scripted band live in this world at all? | about 10 minutes |
| 5 | Validation: four runs (given survival skills with a world model; without sleep-consolidated learning; no given skills; without the world model), 100M agent-steps each, then a 400-day continuous world each | re-estimate from step 4 |
| 6 | Look at the validation | minutes |
| 7 | Core experiment: 18 runs (reward sharing × food × seeds), 100M agent-steps each, each with a continuous world | 18 × the time of one validation run: spread it over sessions |
| 7b | Optional: archetype intervention, 6 runs (one tyrant or one saint among the founders) | about 5–6 hours |
| 7c | Optional: inheritance, 6 continuous worlds where individuals keep learning (children start from a parent's learned or inborn brain) | about 2–4 hours |
| 8 | Report and download | minutes |
| 9 | Full playback of the longest-lived world, every 5-day chunk, playable here | about as long as that world lasted |

Everything is saved to Drive and resumes: after a disconnect, rerun steps 1–3 and the step you were on. Finished runs are skipped; unfinished ones continue from their last checkpoint. Every run records the code version, so runs from different code are never mixed.

## 1. GPU check

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"   # let training subprocesses use the whole GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import jax
print("JAX", jax.__version__, "devices:", jax.devices())
assert jax.default_backend() == "gpu", "No GPU: Runtime > Change runtime type > A100 GPU, then restart."
if "A100" not in str(jax.devices()[0].device_kind):
    print("Note: not an A100. Everything works, but the time estimates above will be longer.")

## 2. Load the code

Upload `lifesim_code.zip` when asked (or set `ZIP_FROM_DRIVE` and run this after step 3).

In [ ]:
import glob, json, shutil, subprocess, sys, zipfile
ZIP_FROM_DRIVE = ""   # e.g. "/content/drive/MyDrive/agon/lifesim_code.zip"
if ZIP_FROM_DRIVE and os.path.exists(ZIP_FROM_DRIVE):
    zip_path = ZIP_FROM_DRIVE
else:
    from google.colab import files
    zip_path = next(iter(files.upload()))
shutil.rmtree("/content/lifesim", ignore_errors=True)
os.makedirs("/content/lifesim")
with zipfile.ZipFile(zip_path) as z:
    z.extractall("/content/lifesim")
if not os.path.exists("/content/lifesim/pyproject.toml"):        # zip made with a top-level folder
    inner = next(os.path.dirname(p) for p in glob.glob("/content/lifesim/*/pyproject.toml"))
    for name in os.listdir(inner):
        shutil.move(os.path.join(inner, name), "/content/lifesim")
os.chdir("/content/lifesim")
!pip install -q -U "flax>=0.12.10" "optax>=0.2.8" pytest
!pip install -q --no-deps -e .
import importlib.metadata as md
v = {p: md.version(p) for p in ("jax", "flax", "optax")}
print(v)
assert tuple(int(x) for x in v["flax"].split(".")[:3]) >= (0, 12, 10), "Flax too old: Runtime > Restart session, rerun this cell."
sys.path.insert(0, "/content/lifesim")
from lifesim.presets import PRESETS, code_version
PRESET = "society30"   # society30: 32 x 32 at half-hour ticks (recommended); society: 250 x 250 at 1-minute ticks
print("code version", code_version(), "| preset", PRESET, PRESETS[PRESET])

## 3. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
# One folder per code version: runs made by different code are never reused or mixed.
# (experiment.py also refuses to reuse a run made by other code.)
ROOT = f"/content/drive/MyDrive/agon/runs/exp_{code_version()}_{PRESET}"   # one folder per code version and preset
os.makedirs(ROOT, exist_ok=True)
print("experiments saved to", ROOT)

## 4. Benchmark

Tries 4, 8, 16, 32, 64, 128 and 256 parallel worlds of the chosen preset (sleep consolidation carries a day of experience, so memory per world is higher than before), stops at the first that runs out of memory, and picks the fastest. Skip it after a reconnect by setting `WORLDS` by hand.

In [ ]:
WORLDS = None   # set a number to skip the benchmark
if WORLDS is None:
    out = subprocess.run([sys.executable, "scripts/benchmark.py", "--preset", PRESET, "--worlds", "4", "8", "16", "32", "64", "128", "256",
                          "--updates", "3", "--target-agent-steps", "1e8"], capture_output=True, text=True)
    print(out.stdout[-2500:] or out.stderr[-2500:])
    rec = [l for l in out.stdout.splitlines() if l.startswith("RECOMMENDED")]
    WORLDS = int(rec[-1].split("--worlds")[1].split()[0]) if rec else 8
print("WORLDS =", WORLDS)

## 4b. Viability check

Hand-scripted agents doing what a competent forager band would (drink and eat early, sleep at night, take cover when cold or hot, stand together against predators, mothers stay with infants), using only the given skills and what each has seen. If they cannot keep their numbers up, no amount of training will. With `society30` (checked before this notebook was updated): every band lasted 100+ days with children born to mothers born there, and 1 of 3 kept its numbers up for the full 4 years (with full map knowledge, 2 of 3); deaths mostly from old age, then starvation. With `society`: 0 of 3, all extinct within 15 days, so do not train it as it stands.

In [ ]:
!python scripts/viability.py --preset {PRESET} --knowledge own --days 160 --seeds 3

## 5. Validation

Four runs, each trained and then lived 400 days in one continuous world (a playback page every 4th 5-day chunk; the inspector shows each person's decisions, with how likely its policy rated each act):
- `caveman`: the design for the core experiment. Agents have given survival skills (seek water, seek food, flee, seek shelter) they can choose like any action, and a world model they imagine with; social life is learned.
- `caveman_no_sleep_learning`: the same, but learning does not wait for sleep.
- `blank`: no given skills, primitive actions only.
- `caveman_no_world_model`: the same as caveman without the world model (no ideas).

Progress lines show lifespan, thirst's share of deaths, births, sleep by night and day, and the share of experience learned.

In [ ]:
VALIDATE_STEPS = 1e8
!python scripts/experiment.py validate --root {ROOT} --preset {PRESET} --worlds {WORLDS} --agent-steps {VALIDATE_STEPS} --playback-every 4

## 6. Look at the validation

What to check before the core experiment:
- **Thirst** should no longer be the main cause of death (it was 60–80% before the fixes).
- **Sleep**: more asleep at night than by day means a daily rhythm was learned.
- **Births** in training (`matings_per_1000` and `births` in the history), and continuous worlds that last and reach later generations. Step 6b checks this.
- **caveman vs blank**: how much the given skills help survival, and whether social behavior differs.
- **caveman vs caveman_no_sleep_learning**: whether waiting for sleep to learn costs or helps.
- **caveman vs caveman_no_world_model**: whether ideas help; in the training log, `world_model.trust_in_ideas` shows whether agents came to act on them. In a playback, the inspector's Ideas tab shows what one imagined.

In [ ]:
import matplotlib.pyplot as plt
!python scripts/experiment.py report --root {ROOT}

runs = {n: json.load(open(f"{ROOT}/validate/{n}/history.json"))["history"]
        for n in ("caveman", "caveman_no_sleep_learning", "blank", "caveman_no_world_model") if os.path.exists(f"{ROOT}/validate/{n}/history.json")}
share = lambda r, k: r["deaths"].get(k, 0) / max(1, sum(r["deaths"].values()))
nan = lambda y: float("nan") if y is None else y
panels = [("Mean age at death (ticks)", lambda r: r["mean_age_at_death"]),
          ("Thirst's share of deaths", lambda r: share(r, "thirst")),
          ("Births per window", lambda r: r.get("births")),
          ("Time with someone in view", lambda r: r.get("in_company_share")),
          ("Asleep at night", lambda r: r.get("asleep_share_night")),
          ("Asleep by day", lambda r: r.get("asleep_share_day")),
          ("Share of experience learned", lambda r: r.get("learned_share")),
          ("Contact aggression per 1,000 agent-ticks", lambda r: r["per_1000_agent_ticks"]["strikes_on_agents"])]
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, (title, f) in zip(axes.flat, panels):
    for name, hist in runs.items():
        ax.plot([r["agent_steps"] / 1e6 for r in hist], [nan(f(r)) for r in hist], label=name)
    ax.set_title(title, fontsize=10); ax.set_xlabel("million agent-steps")
axes[0, 0].legend(); plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
for name in runs:
    f = f"{ROOT}/validate/{name}/continuous/summary.json"
    if not os.path.exists(f):
        continue
    S = json.load(open(f)); days = [5 * (s["chunk"] + 1) for s in S]
    axes[0].plot(days, [s["population"]["alive"] for s in S], label=name)
    axes[1].plot(days, [s["births_so_far"] for s in S], label=name)
    axes[2].plot(days, [s["population"]["max_generation"] or 0 for s in S], label=name)
    axes[3].plot(days, [s["sleep"]["asleep_share_night"] - s["sleep"]["asleep_share_day"] for s in S], label=name)
for ax, t in zip(axes, ["Alive", "Births so far", "Latest generation", "Sleep: night minus day share"]):
    ax.set_title(t); ax.set_xlabel("day")
axes[0].legend(); plt.tight_layout(); plt.show()

In [ ]:
# A playback page from the continuous world (change RUN or pick another page from the list)
import html
from IPython.display import HTML, display
RUN = "caveman"
pages = sorted(glob.glob(f"{ROOT}/validate/{RUN}/continuous/chunk_*.html"))
print([os.path.basename(p) for p in pages])
if pages:
    display(HTML(f'<iframe srcdoc="{html.escape(open(pages[0]).read())}" style="width:100%;height:1100px;border:0"></iframe>'))

In [ ]:
# Cause and effect: the continuous world told as stories (what led to what), and every death
# traced back to its first cause. Options: --agent K, --from-day, --to-day, --all
RUN = "caveman"
!python scripts/causal_story.py {ROOT}/validate/{RUN}/continuous --max 40

In [ ]:
# Technology: which building blocks of know-how appeared, who first, how far they spread, and which
# personalities did which skills (Spearman, permutation p, false-discovery corrected)
!python scripts/tech_tree.py {ROOT}/validate/{RUN}

## 6b. Persistence gate

Did anyone born in the `caveman` agents' continuous world grow up (15 days)? If not, the trained agents cannot outlast their founders, and every later experiment would study worlds that simply die out: `experiment.py` refuses to start core, archetypes, inheritance and stress until this passes (pass `--skip-gate` to run them anyway).

In [ ]:
!python scripts/experiment.py gate --root {ROOT} --preset {PRESET}

## 7. Core experiment

Reward sharing α ∈ {0, 0.5, 1} × food {1.0, 0.4} × seeds {0, 1, 2} = 18 runs of the same setup, 100M agent-steps each (the first A100 run plateaued around 60–100M), each followed by a 400-day continuous world (a playback page every 8th 5-day chunk). Resumes run by run: rerun this cell after a disconnect.

To spread it over several sessions, set `ONLY` to a list of run numbers (0–17), e.g. `range(0, 6)` today and `range(6, 12)` tomorrow.

In [ ]:
CORE_STEPS = 1e8
ONLY = None   # None = all 18; or e.g. range(0, 6)
if ONLY is None:
    !python scripts/experiment.py core --root {ROOT} --preset {PRESET} --worlds {WORLDS} --agent-steps {CORE_STEPS} --playback-every 8
else:
    for k in ONLY:
        !python scripts/experiment.py core --root {ROOT} --preset {PRESET} --worlds {WORLDS} --agent-steps {CORE_STEPS} --playback-every 8 --only {k}

## 7b. Optional: archetype intervention

One tyrant or one saint among the founders (α = 0, normal food, 3 seeds each). The control is the core experiment's `a0.0_f1.0_s0`–`s2`. Archetypes set feelings and capacities, not behavior: whether the tyrant becomes one depends on what it learns. The report below shows which role each one actually took.

In [ ]:
ARCH_STEPS = 1e8
!python scripts/experiment.py archetypes --root {ROOT} --preset {PRESET} --worlds {WORLDS} --agent-steps {ARCH_STEPS} --playback-every 8
!python scripts/compare_runs.py {ROOT}/core/a0.0_f1.0_s* {ROOT}/archetypes/*

## 7c. Optional: inheritance (Lamarckian or Darwinian)

The validated `caveman` agents live on in continuous worlds where every individual keeps learning from its own life. Newborns start from a parent's brain as it is now (`parent`) or as the parent was born with it (`innate`): under `innate`, what a parent learned can pass on only by being watched. 3 seeds each. Needs step 5's `caveman` run.

In [ ]:
!python scripts/experiment.py inheritance --root {ROOT} --preset {PRESET} --playback-every 8

## 7d. Culture: replicate worlds

The validated `caveman` agents in 8 continuous worlds on **one map with the same founders**, differing only in what happens: 4 where they learn by watching (`social`), 4 where they cannot (`solitary`). Inheritance is innate, so learning cannot pass down through the brain: whatever persists across generations in `social` but not in `solitary` passed by watching. Needs the gate (6b).

The analysis asks whether worlds differ in behaviour, beyond inherited traits; which behaviours are customary in some worlds and absent in others; whether a world's ways recur among those born later; and whether speech drifts into dialects.

In [ ]:
REPLICATES = 4
!python scripts/experiment.py culture --root {ROOT} --preset {PRESET} --replicates {REPLICATES} --playback-every 8
!python scripts/culture.py {ROOT}/culture/social_* --against {ROOT}/culture/solitary_* --json {ROOT}/culture/culture.json

## 7e. Placed archetypes

One founder of the validated agents' world is given an archetype's temperament (17 of them, `lifesim/archetypes.py`), and everyone keeps learning as they live; 2 seeds each, against the same world with none. The report shows where each archetype stood: dominance rank, leadership, children. Set `WHICH` to a few (e.g. `"chief,warrior,enforcer,sage"`) to save time: all 17 is 36 worlds.

In [ ]:
WHICH = "all"
!python scripts/experiment.py placed --root {ROOT} --preset {PRESET} --which {WHICH} --playback-every 8

## 7f. Hierarchy in any world

Dominance from contact aggression (exploratory acts left out) and supplants, linearity, steepness, stability, what rank brings, and who leads (`scripts/hierarchy.py`).

In [ ]:
!python scripts/hierarchy.py {ROOT}/validate/caveman

## 8. Report and download

The report includes, for each continuous world, the share of deaths in winter, and for each experiment the roles agents actually fell into (lives grouped by what they did) with where any injected archetype landed.

In [ ]:
!python scripts/experiment.py report --root {ROOT}

In [ ]:
# Histories, logs, continuous-world summaries and births; set PLAYBACKS = True to include the
# playback pages too (about 2 MB each). Checkpoints and trained networks stay in Drive.
PLAYBACKS = False
zip_out = "/content/agon_experiments.zip"
with zipfile.ZipFile(zip_out, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{ROOT}/**/*", recursive=True):
        name = os.path.basename(f)
        if os.path.isdir(f) or name in ("checkpoint.pkl", "params.msgpack", "population.pkl"):
            continue
        if name.endswith(".html") and not PLAYBACKS:
            continue
        z.write(f, os.path.relpath(f, os.path.dirname(ROOT)))
print(f"{os.path.getsize(zip_out) / 1e6:.1f} MB")
from google.colab import files
files.download(zip_out)

## 9. Full playback of the longest-lived world

Experiments record a playback page only every few chunks. This finds the run whose continuous world lasted longest (then most births), lives that world again with the same seed while recording every 5-day chunk, and plays it below: start from the index, and each page links to the previous and next five days. Below the chunks, the index lists every life: born, died and of what, how long it lived, generation, archetype, children, and its conditions with the age each arrived. In a playback, click any agent for its inspector (conditions included; hover one for what it does and how rare it is). Set `RUN` to watch a particular run instead.

In [ ]:
RUN = None    # e.g. f"{ROOT}/core/a0.0_f1.0_s1"
args = ["--run", RUN] if RUN else ["--root", ROOT]
out = subprocess.run([sys.executable, "scripts/full_playback.py", *args], capture_output=True, text=True)
print("\n".join(l for l in out.stdout.splitlines() if not l.startswith("{"))[-2000:] or out.stderr[-2000:])
FULL = next(l.split(": ", 1)[1] for l in out.stdout.splitlines() if l.startswith("full playback: ")).rsplit("/", 1)[0]
print("pages in", FULL)

In [ ]:
# Play it here: a small web server inside this session, so the page-to-page links work
import time
from google.colab import output
server = subprocess.Popen([sys.executable, "-m", "http.server", "8765", "--directory", FULL],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)
output.serve_kernel_port_as_iframe(8765, path="/index.html", height=1100)

In [ ]:
# Or download the whole playback and open index.html in your browser
zip_full = "/content/full_playback.zip"
with zipfile.ZipFile(zip_full, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"{FULL}/*"):
        z.write(f, os.path.basename(f))
print(f"{os.path.getsize(zip_full) / 1e6:.1f} MB")
from google.colab import files
files.download(zip_full)